In [1]:
import sys
sys.path.append('../src') # include the src directory


In [ ]:
import pandas as pd
from transformers import AutoTokenizer

from data_splitter import DataSplitter, TrainTestSplit, StratifiedKFoldSplit
from dataset_builder import TrainTestConverter, FoldsConverter
from model_building import ModelBuilder, BERTClassificationStrategy


In [6]:
file_path = '../data/processed/amyloid-02-07-2025.csv'
df = pd.read_csv(file_path)
df = df.dropna(subset=['Abstract']).reset_index(drop=True)  
df['rejection'] = df['rejection'].map({'Rejected': 0, 'Useful': 1})


In [6]:
df.head()

,PMID,rejection,reason,decision,Title,Abstract,Authors,Journal,References
0,39441361,0,(Pre)Clinical trials. No interaction or amyloi...,NaN,Monoclonal anti-amyloid antibody treatment: th...,The development of monoclonal anti-amyloid ant...,"['Lee S', 'Stögmann E']",Wiener klinische Wochenschrift,"['36449413', '33720637', '35652476', '31205802..."
1,39438925,0,Not enough experimental data,read whole paper,Blockade of brain alkaline phosphatase efficie...,Alzheimer's disease (AD) is the most prevalent...,"['Soria-Tobar L', 'Román-Valero L', 'Sebastián...",Alzheimer's research & therapy,"['11520930', '11274343', '12824062', '9530504'..."
2,39438516,0,"The interactor is not an Ab, Unknown antibody ...",NaN,Thioflavin-T: application as a neuronal body a...,Thioflavin-T (THT) is a common and indispensab...,"['Min JH', 'Sarlus H', 'Oasa S', 'Harris RA']",Scientific reports,"['20399286', '30108983', '28280572', '2729542'..."
3,39434125,0,There are no interactions described,NaN,Amyloid-β (Aβ) immunotherapy induced microhemo...,Anti-amyloid-β (Aβ) immunotherapy trials have ...,"['Taylor X', 'Noristani HN', 'Fitzgerald GJ', ...",Molecular neurodegeneration,"['3159021', '33720637', '36449413', '35542991'..."
4,39426463,0,Not enough experimental data,NaN,Early divergent modulation of NLRP2's and NLRP...,"Alzheimer's disease (AD), the most prevalent h...","['Chiarini A', 'Armato U', 'Gui L', 'Yin M', '...",Brain research,[]


In [7]:
target = 'rejection'
splitter = DataSplitter(TrainTestSplit(test_size=0.2))
folds = splitter.split(df, target)

2025-07-09 22:40:42,955 - INFO - Splitting data using the selected strategy.
2025-07-09 22:40:42,956 - INFO - Performing stratified train-test split.
2025-07-09 22:40:42,963 - INFO - Train-test split completed.


In [10]:
model_name = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:

SPECIAL_TOKEN = "[KEY]"

tokenizer.add_special_tokens({
"additional_special_tokens": [SPECIAL_TOKEN]
})

1

In [13]:
keywords = ["aggregates", "amyloid", "scfv"]
tts_converter = TrainTestConverter()
hf_dataset = tts_converter.convert(folds, tokenizer, keywords)


Map: 100%|██████████| 282/282 [00:00<00:00, 5477.67 examples/s]
2025-07-09 22:41:23,560 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.


In [14]:
hf_dataset

DatasetDict({
    train: Dataset({
        features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
        num_rows: 1126
    })
    test: Dataset({
        features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
        num_rows: 282
    })
})

In [16]:
splitter.set_strategy(StratifiedKFoldSplit(n_splits=5, shuffle=True, random_state=42))
folds = splitter.split(df, target)

2025-07-09 22:41:41,738 - INFO - Switching data splitting strategy.
2025-07-09 22:41:41,739 - INFO - Splitting data using the selected strategy.
2025-07-09 22:41:41,740 - INFO - Performing Stratified K-Fold split (5 folds)
2025-07-09 22:41:41,747 - INFO - Stratified K-Fold split completed.


In [17]:
cv_converter = FoldsConverter(tts_converter)
hf_cv_dataset = cv_converter.convert(folds, tokenizer, keywords)

Map: 100%|██████████| 282/282 [00:00<00:00, 5356.83 examples/s]
2025-07-09 22:41:59,393 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 22:41:59,394 - INFO - Fold 0: conversion finished.
Map: 100%|██████████| 282/282 [00:00<00:00, 2307.37 examples/s]
2025-07-09 22:41:59,730 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 22:41:59,730 - INFO - Fold 1: conversion finished.
Map: 100%|██████████| 282/282 [00:00<00:00, 5884.66 examples/s]
2025-07-09 22:41:59,991 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 22:41:59,991 - INFO - Fold 2: conversion finished.
Map: 100%|██████████| 281/281 [00:00<00:00, 5770.74 examples/s]
2025-07-09 22:42:00,256 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 22:42:00,256 - INFO - Fold 3: conversion finished.
Map: 100%|██████

In [12]:
hf_cv_dataset

[DatasetDict({
     train: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
         num_rows: 1126
     })
     test: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
         num_rows: 282
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
         num_rows: 1126
     })
     test: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_m

In [18]:
hf_dataset

DatasetDict({
    train: Dataset({
        features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
        num_rows: 1126
    })
    test: Dataset({
        features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
        num_rows: 282
    })
})

In [19]:
import torch 
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
device

device(type='mps')

In [20]:
strategy = BERTClassificationStrategy(
    model_name=model_name,
    special_tokens=["[KEY]"],
    epochs=1,
    freeze_last_k_layers=1,
    device=device,
)

builder = ModelBuilder(strategy)
model = builder.build_model(hf_dataset)

2025-07-09 22:42:56,926 - INFO - Building and training the model on single dataset.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cambridgeltl/SapBERT-from-PubMedBERT-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2025-07-09 22:42:58,557 - INFO - Adding special tokens and resizing model embeddings.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
2025-07-09 22:42:58,761 - INFO - Freezing model layers, except pooler, classifier, last K encoder layers, and embeddings.
2025-07-09 22:42:58,777 - INFO - Starting training.
/Users/sonor/Documents/Projekty/deep-skim/.venv/lib/python3.11/site-packages/torc

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.633021,0.730496,0.191011,0.809524,0.309091


In [21]:
model['trainer'].evaluate(hf_dataset['test'])

/Users/sonor/Documents/Projekty/deep-skim/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.6330209970474243,
 'eval_accuracy': 0.7304964539007093,
 'eval_precision': 0.19101123595505617,
 'eval_recall': 0.8095238095238095,
 'eval_f1': 0.3090909090909091,
 'eval_runtime': 10.6092,
 'eval_samples_per_second': 26.581,
 'eval_steps_per_second': 1.697,
 'epoch': 1.0}